# 03 - Model Training & Evaluation
## InclusionScore AI - Alternate Credit Scoring

**Models:** Logistic Regression (baseline) and XGBoost (primary)  
**Metrics:** AUC-ROC, Precision, Recall, F1-Score, Brier Score  
**Data:** Processed train/val/test splits from Phase 2

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, roc_auc_score
from sklearn.calibration import calibration_curve

from src.features import load_processed
from src.models import (
    train_baseline, train_xgboost, evaluate,
    save_model, load_model,
)
from src.config import RANDOM_SEED

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Processed Splits

In [ ]:
X_train, y_train = load_processed('train')
X_val,   y_val   = load_processed('val')
X_test,  y_test  = load_processed('test')

print(f'Train: {X_train.shape}  default rate: {y_train.mean():.3f}')
print(f'Val  : {X_val.shape}  default rate: {y_val.mean():.3f}')
print(f'Test : {X_test.shape}  default rate: {y_test.mean():.3f}')

## 2. Baseline: Logistic Regression

In [ ]:
lr_model, lr_metrics = train_baseline(X_train, y_train, X_val, y_val)
lr_test_metrics = evaluate(lr_model, X_test, y_test, label='LR (test)')

## 3. XGBoost Training

In [ ]:
xgb_model, xgb_val_metrics = train_xgboost(X_train, y_train, X_val, y_val)
xgb_test_metrics = evaluate(xgb_model, X_test, y_test, label='XGBoost (test)')

## 4. Save Model Artifact

In [ ]:
save_model(
    xgb_model, xgb_test_metrics,
    X_train_rows=len(X_train),
    X_val_rows=len(X_val),
)

## 5. ROC Curve

In [ ]:
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_proba_lr  = lr_model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, style in [('XGBoost', y_proba_xgb, '-'), ('Logistic Regression', y_proba_lr, '--')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, style, label=f'{name} (AUC={auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Precision-Recall Curve

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba, style in [('XGBoost', y_proba_xgb, '-'), ('Logistic Regression', y_proba_lr, '--')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ax.plot(rec, prec, style, label=name)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Confusion Matrix

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred_xgb)
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=ax,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - XGBoost (Test Set)')
plt.tight_layout()
plt.show()
print(f'\nTrue Positives:  {cm[1,1]:,}')
print(f'False Positives: {cm[0,1]:,}')
print(f'True Negatives:  {cm[0,0]:,}')
print(f'False Negatives: {cm[1,0]:,}')

## 8. Calibration Curve & Brier Score

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for name, proba in [('XGBoost', y_proba_xgb), ('Logistic Regression', y_proba_lr)]:
    prob_true, prob_pred = calibration_curve(y_test, proba, n_bins=10, strategy='uniform')
    ax.plot(prob_pred, prob_true, 's-', label=name)
ax.plot([0, 1], [0, 1], 'k:', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve (Test Set)')
ax.legend()
plt.tight_layout()
plt.show()

from sklearn.metrics import brier_score_loss
print(f'Brier Score - XGBoost: {brier_score_loss(y_test, y_proba_xgb):.4f}')
print(f'Brier Score - LR:      {brier_score_loss(y_test, y_proba_lr):.4f}')

## 9. Feature Importance

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
importances = pd.Series(xgb_model.feature_importances_, index=X_test.columns)
importances.sort_values().plot(kind='barh', ax=ax, color='#2196F3')
ax.set_title('XGBoost Feature Importance (gain)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 10. Summary

| Model | AUC-ROC | Precision | Recall | F1 | Brier |
|-------|---------|-----------|--------|------|-------|
| Logistic Regression | ~0.857 | ~0.211 | ~0.766 | ~0.331 | ~0.148 |
| **XGBoost** | **~0.860** | **~0.232** | **~0.737** | **~0.352** | **~0.128** |

XGBoost is the primary model, saved as `models/xgb_v1.joblib`.